This is the Update file for the heroes table. All you need to do is run all cells to update the table and hero-pages folder to include the most recent data from Fire Emblem Heroes.

In [5]:
import pandas as pd
import bs4
import requests
import plotly.express as px
import kaleido
import regex
import time
import os
import lxml
import httpx

In [3]:
heroes_table = pd.read_csv('heroes_table.csv').drop(columns=['Unnamed: 0'])

In [ ]:
heroes = requests.get('https://feheroes.fandom.com/wiki/Level_40_stats_table')

In [ ]:
heroes = bs4.BeautifulSoup(heroes.text)

In [ ]:
rows = heroes.find_all('tr', attrs={'class':'hero-filter-element'})
# Gets all rows of Heroes as a list


In [ ]:
links = [] # This cell creates the links for each individual heroes' info pages
for i in range(len(rows)):
    links.append("https://feheroes.fandom.com" + rows[i].find_all('a')[0].get('href'))
links

In [ ]:
heroes_table = pd.DataFrame(columns=['Hero', 'Entry', 'Move', 'Weapon', 'HP', 'Atk', 'Spd', 'Def', 'Res', 'Total'])
for i in range(len(rows)):
    row = rows[i].find_all('td')
    heroes_table.loc[len(heroes_table)] = [
        row[0].find('a').get('title'),
        row[2].find('img').get('alt'),
        row[3].find('img').get('alt'),
        row[4].find('img').get('alt'),
        int(row[5].text),
        int(row[6].text),
        int(row[7].text),
        int(row[8].text),
        int(row[9].text),
        int(row[10].text)
    ]
heroes_table['Link'] = links
heroes_table # This heroes table only includes data that is available on the Lv. 40 Stats Table on the Fandom page

In [ ]:
def color(str):
    return str.split(' ')[0]

heroes_table['Color'] = heroes_table['Weapon'].apply(color)
heroes_table.head()
# This bit just adds the color of the unit by pulling from the Weapon column

Next is adding book information by pulling data from hero pages stored in the hero-pages directory.

In [18]:
test = bs4.BeautifulSoup(requests.get(heroes_table['Link'].iloc[1]).text)
float(test.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text)
# Test is here to find where book info is stored.

5.9

In [48]:
pagedir = os.listdir('hero-pages') # Current hero pages in hero-pages directory
pagedir[:10]
with open('hero-pages/AbelThePanther.html', 'r', encoding="UTF-8") as f:
    soup = bs4.BeautifulSoup(f, 'lxml')
    print(float(soup.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text))


1.0


In [5]:
versions = []
for i in range(heroes_table.shape[0]):
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)

    with open(f'hero-pages/{name}.html', 'r', encoding="UTF-8") as f:
        soup = bs4.BeautifulSoup(f.read(), 'lxml')
        versions.append(float(soup.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text))
    
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [6]:
# Add column
heroes_table['Version'] = versions
heroes_table.head()

,Hero,Entry,Move,Weapon,HP,Atk,Spd,Def,Res,Total,Link,Color,Version,Book
0,Abel: The Panther,Shadow Dragon / (New) Mystery,Cavalry,Blue Lance,39,33,32,25,25,154,https://feheroes.fandom.com/wiki/Abel:_The_Pan...,Blue,1.0,1
1,Aelfric: Custodian Monk,Three Houses,Infantry,Red Tome,43,40,31,20,37,171,https://feheroes.fandom.com/wiki/Aelfric:_Cust...,Red,5.9,5
2,Alcryst: Tender Archer,Engage,Infantry,Colorless Bow,39,43,46,30,24,182,https://feheroes.fandom.com/wiki/Alcryst:_Tend...,Colorless,7.7,7
3,Alcryst: Tender Groom,Engage,Infantry,Red Bow,40,44,46,35,23,188,https://feheroes.fandom.com/wiki/Alcryst:_Tend...,Red,8.5,8
4,Alear: Awoken Divinity,Engage,Infantry,Green Breath,40,41,47,33,37,198,https://feheroes.fandom.com/wiki/Alear:_Awoken...,Green,7.8,7


In [7]:
# Now use that Version column to create a Book column
def book(ver):
    return int(ver)

heroes_table['Book'] = heroes_table['Version'].apply(book)
heroes_table.tail()

,Hero,Entry,Move,Weapon,HP,Atk,Spd,Def,Res,Total,Link,Color,Version,Book
1295,Zephia: Scheming Dragon,Engage,Flying,Red Tome,41,40,44,23,24,172,https://feheroes.fandom.com/wiki/Zephia:_Schem...,Red,7.70,7
1296,Zephiel: The Liberator,The Binding Blade,Armored,Red Sword,55,35,16,38,24,168,https://feheroes.fandom.com/wiki/Zephiel:_The_...,Red,1.20,1
1297,Zephiel: Winter's Crown,The Blazing Blade,Armored,Red Sword,48,38,19,38,36,179,https://feheroes.fandom.com/wiki/Zephiel:_Wint...,Red,4.00,4
1298,Zihark: Ninja Blademaster,Path of Radiance,Infantry,Red Sword,43,37,42,30,24,176,https://feheroes.fandom.com/wiki/Zihark:_Ninja...,Red,4.11,4
1299,Þjazi: Ruthless Jötun,Heroes,Armored,Green Axe,49,50,18,47,42,206,https://feheroes.fandom.com/wiki/%C3%9Ejazi:_R...,Green,8.11,8


In [ ]:
# 47m 3.2s runtime

# def version(link):
#     time.sleep(0.5)
#     html = bs4.BeautifulSoup(requests.get(link).text)
#     return float(html.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text)

# heroes_table['Version'] = heroes_table['Link']
# heroes_table['Version'] = heroes_table['Version'].apply(version)

# Obsolete code. Very inefficient way to pull version info for each unit for Version Column

In [ ]:
# heroes_pages = heroes_table[['Hero', 'Link']]
# heroes_pages.head()
# Hero and Link table

In [ ]:
# runtime 50m 5.8s

# def grabber(link):
#     time.sleep(0.5)
#     print(link.split('/')[-1])
#     return bs4.BeautifulSoup(requests.get(link).text)

# heroes_pages['Pages'] = heroes_pages['Link'].apply(grabber)

# Another obsolete bit of code that essentially adds the entire HTML page of each hero in a Page column.

In [11]:
test = bs4.BeautifulSoup(requests.get(heroes_table['Link'].iloc[0]).text)
test.find('time').text
# Test is here to find where date added is stored.

'2017-02-02'

In [19]:
releaseDate = []
for i in range(heroes_table.shape[0]):
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)

    with open(f'hero-pages/{name}.html', 'r', encoding="UTF-8") as f:
        soup = bs4.BeautifulSoup(f.read(), 'lxml')
        releaseDate.append(soup.find('time').text.strip())
    
    print(name)

AbelThePanther
AelfricCustodianMonk
AlcrystTenderArcher
AlcrystTenderGroom
AlearAwokenDivinity
AlearDragonChild
AlearDragonYouth
AlearEngagingFire
AlearFellDragonChild
AlearGiftedDragons
AlearRedDaysPast
AlearSeasideDragon
AlfonseAskranDuo
AlfonseHeirtoOpenness
AlfonsePrinceofAskr
AlfonseSpringPrince
AlfonseUpliftingLove
AlfredFloralProtector
AlmHeroofProphecy
AlmImperialAscent
AlmLovebirdDuo
AlmPowerfulResolve
AlmSaint-King
AltenaLuminousRider
AltinaCross-TimeDuo
AltinaDawn'sTrueblade
AltinaUnrivaledDawn
AmeliaBlossomingTalent
AmeliaRoseoftheWar
AnankosSeethingSilence
AnnaCommander
AnnaSecretCharmer
AnnaSecretSeller
AnnaTwicetheAnna
AnnaWealth-Wisher
AnnandKnight-Defender
AnnetteFestiveHelper
AnnetteOverachiever
ArdenStrongandTough
AresBlackKnight
AreteRequiem'sBeauty
ArionGungnir'sHeir
ArlenMageofKhadein
ArthurFuriousMage
ArthurHaplessHero
ArturSilverSaint
ArvalCycleKeeper
ArvisEmperorofFlame
AsbelWindsweptYouth
AshBorrowedPower
AshEarnestGreetings
AshRetainertoAskr
AsheBuddingChival

In [20]:
heroes_table['ReleaseDate'] = releaseDate

In [21]:
# heroes_table = heroes_table.drop(columns=['Link'])
heroes_table.to_csv('heroes_table.csv') # I'll keep the link column in, just as a reference.

##### Below is code that essentially pulls each hero's info page as an HTML file and adds it to the hero-pages folder. This is done to keep the file size of the heroes table to a minimum, as well as keeping a local reference for each page that makes accessing these pages quicker and more efficient. The hero-pages folder is gitignored.

In [7]:
heroes_table = pd.read_csv('heroes_table.csv').drop(columns=['Unnamed: 0'])
pagedir = os.listdir('hero-pages') # Current hero pages in hero-pages directory

In [8]:
missingidx = []
for i in range(heroes_table.shape[0]):
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)

    if f'{name}.html' not in pagedir:
        print(heroes_table['Hero'].iloc[i])
        missingidx.append(i)

# Use this to check which pages are currently not in directory

Rennac: Rich "Merchant"
Reyson: White Prince
Rhajat: Black Magician
Rhea: Aura of Love
Rhea: Immaculate One
Rhea: Loving Matriarch
Rhea: Witch of Creation
Rhys: Gentle Basker
Rickard: Carefree Culprit
Ricken: Shepherd Novice
Riev: Blood Beryl
Rinea: Reminiscent Belle
Rinkah: Consuming Flame
Rinkah: Scion of Flame
Robin: Exalt's Deliverer
Robin: Exalt's Other Half
Robin: Exalt's Right Hand
Robin: Fall Reincarnation
Robin: Fall Vessel
Robin: Fated Vessel
Robin: Fell Reincarnation
Robin: Fell Tactician
Robin: Fell Vessel
Robin: Festive Tactician
Robin: High Deliverer
Robin: Keen Groom
Robin: Mystery Tactician
Robin: Seaside Tactician
Robin: Tactful Deliverer
Robin: Vessels of Fate
Roderick: Steady Squire
Rolf: Tricky Archer
Ronan: Villager of Iz
Rosado: Adorable Artist
Roshea: Coyote's Faithful
Ross: His Father's Son
Roy: Blazing Bachelors
Roy: Blazing Lion
Roy: Brave Lion
Roy: Young Lion
Roy: Youthful Gifts
Rudolf: Emperor of Rigel
Rune: Source of Wisdom
Rutger: Lone Swordsman
Ryoma: Dan

The missingidx list created above can be used to add the missing hero pages to the hero-pages folder.

In [44]:
for i in range(heroes_table.shape[0]):
    # Create file name
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)
    
    # Pull HTML from site
    time.sleep(0.5)
    soup = bs4.BeautifulSoup(requests.get(heroes_table['Link'].iloc[i]).content, "lxml")
    script = soup.find_all('script')
    [s.decompose() for s in script]
    navbox = soup.find('div', class_= 'navbox')
    navbox.decompose()
    aside = soup.find_all('aside')
    [a.decompose() for a in aside]
    commhead = soup.find('div', class_='community-header-wrapper')
    commhead.decompose()
    ul = soup.find_all('ul', class_='wds-tabs')
    [u.decompose() for u in ul]
    foot = soup.find('footer', class_='global-footer')
    foot.decompose()
    pagef = soup.find('div', class_='page-footer')
    pagef.decompose()
    sidew = soup.find('div', class_='page-side-tools__wrapper')
    sidew.decompose()
    pheadt = soup.find('div', class_='page-header__top')
    pheadt.decompose()
    item1 = soup.find('div', class_='global-explore-navigation')
    item1.decompose()
    navtop = soup.find('div', class_='community-navigation fandom-sticky-header')
    navtop.decompose()
    ads = soup.find('div', class_='bottom-ads-container')
    ads.decompose()
    navi = soup.find('nav', class_='global-top-navigation')
    navi.decompose()
    act = soup.find('div', class_='page-header__actions')
    act.decompose()
    [svg.decompose() for svg in soup.find_all('svg')]
    soup.find('div', class_='toc').decompose()

    with open(F"hero-pages/{name}.html", "w", encoding = 'utf-8') as file:
        file.write(str(soup.prettify()))
    print(name)


AbelThePanther
AelfricCustodianMonk
AlcrystTenderArcher
AlcrystTenderGroom
AlearAwokenDivinity
AlearDragonChild
AlearDragonYouth
AlearEngagingFire
AlearFellDragonChild
AlearGiftedDragons
AlearRedDaysPast
AlearSeasideDragon
AlfonseAskranDuo
AlfonseHeirtoOpenness
AlfonsePrinceofAskr
AlfonseSpringPrince
AlfonseUpliftingLove
AlfredFloralProtector
AlmHeroofProphecy
AlmImperialAscent
AlmLovebirdDuo
AlmPowerfulResolve
AlmSaint-King
AltenaLuminousRider
AltinaCross-TimeDuo
AltinaDawn'sTrueblade
AltinaUnrivaledDawn
AmeliaBlossomingTalent
AmeliaRoseoftheWar
AnankosSeethingSilence
AnnaCommander
AnnaSecretCharmer
AnnaSecretSeller
AnnaTwicetheAnna
AnnaWealth-Wisher
AnnandKnight-Defender
AnnetteFestiveHelper
AnnetteOverachiever
ArdenStrongandTough
AresBlackKnight
AreteRequiem'sBeauty
ArionGungnir'sHeir
ArlenMageofKhadein
ArthurFuriousMage
ArthurHaplessHero
ArturSilverSaint
ArvalCycleKeeper
ArvisEmperorofFlame
AsbelWindsweptYouth
AshBorrowedPower
AshEarnestGreetings
AshRetainertoAskr
AsheBuddingChival

# TESTING ZONE

In [42]:
soup = bs4.BeautifulSoup(requests.get("https://feheroes.fandom.com/wiki/Abel:_The_Panther").content, "lxml")
script = soup.find_all('script')
[s.decompose() for s in script]
navbox = soup.find('div', class_= 'navbox')
navbox.decompose()
aside = soup.find_all('aside')
[a.decompose() for a in aside]
commhead = soup.find('div', class_='community-header-wrapper')
commhead.decompose()
ul = soup.find_all('ul', class_='wds-tabs')
[u.decompose() for u in ul]
foot = soup.find('footer', class_='global-footer')
foot.decompose()
pagef = soup.find('div', class_='page-footer')
pagef.decompose()
sidew = soup.find('div', class_='page-side-tools__wrapper')
sidew.decompose()
pheadt = soup.find('div', class_='page-header__top')
pheadt.decompose()
item1 = soup.find('div', class_='global-explore-navigation')
item1.decompose()
navtop = soup.find('div', class_='community-navigation fandom-sticky-header')
navtop.decompose()
ads = soup.find('div', class_='bottom-ads-container')
ads.decompose()
navi = soup.find('nav', class_='global-top-navigation')
navi.decompose()
act = soup.find('div', class_='page-header__actions')
act.decompose()
[svg.decompose() for svg in soup.find_all('svg')]
soup.find('div', class_='toc').decompose()

with open(F"test/test.html", "w", encoding = 'utf-8') as file:
    file.write(str(soup.prettify()))

# Construction Below

In [3]:
heroes = requests.get('https://feheroes.fandom.com/wiki/Level_40_stats_table')
heroes = bs4.BeautifulSoup(heroes.text, features='lxml')
print(heroes)

<!DOCTYPE html>
<html lang="en-US"><head><title>Just a moment...</title><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/><meta content="IE=Edge" http-equiv="X-UA-Compatible"/><meta content="noindex,nofollow" name="robots"/><meta content="width=device-width,initial-scale=1" name="viewport"/><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,"Helvetica Neue",Arial,"Noto Sans",sans-serif,"Apple Color Emoji","Segoe UI Emoji","Segoe UI Symbol","Noto Color Emoji"}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;padding-left:1.5rem;max-width:60rem}@media (width <= 720px){.main-content{margin-top:4rem}}#challenge-error-text{background-image:url("");background-repeat:no-repeat;background-size:contain;padding-left:34px}</style><meta content="360" http-equiv="refresh"/></head><body><div clas

In [16]:
headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:149.0) Gecko/20100101 Firefox/149.0'}
heroes = httpx.get('https://feheroes.fandom.com/wiki/Level_40_stats_table', headers=headers)

In [17]:
heroes.encoding

'utf-8'

In [19]:
with open('lv40_heroes.html', 'w', encoding='utf-8') as file:
    file.write(heroes.text)